## Model Training & Selection — Customer Churn

### By:
Data Science Team

### Date:
2026-02-25

### Description:
Train multiple classification models (baseline → advanced), compare them with
stratified cross-validation, select the best one, and serialise it to
`data/06_models/`.

**Pipeline:**
1. Load processed data from `data/05_model_input/`  
2. Train: Logistic Regression (baseline), Random Forest, Gradient Boosting  
3. Compare with `cross_val_score` (AUC-ROC, F1, Precision, Recall)  
4. Save best model + metadata to `data/06_models/`  
5. Save predictions to `data/07_model_output/`

## 📚 Import libraries

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import sklearn as sk
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

print("Python  :", sys.version)
print("Pandas  :", pd.__version__)
print("sklearn :", sk.__version__)

Python  : 3.12.12 (main, Feb 12 2026, 00:42:14) [Clang 21.1.4 ]
Pandas  : 2.3.3
sklearn : 1.8.0


## 💾 Load processed data

In [2]:
DATA_DIR = Path.cwd().resolve().parents[1] / "data"
MODELS_DIR = DATA_DIR / "06_models"
OUTPUT_DIR = DATA_DIR / "07_model_output"
INPUT_DIR = DATA_DIR / "05_model_input"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

x_train = pd.read_parquet(INPUT_DIR / "x_train.parquet")
x_test = pd.read_parquet(INPUT_DIR / "x_test.parquet")
y_train = pd.read_parquet(INPUT_DIR / "y_train.parquet").squeeze()
y_test = pd.read_parquet(INPUT_DIR / "y_test.parquet").squeeze()

with open(MODELS_DIR / "feature_names.json") as f:
    feature_names = json.load(f)

print(f"Train: {x_train.shape}  |  Test: {x_test.shape}")
print(f"Churn rate — train: {y_train.mean():.2%}  |  test: {y_test.mean():.2%}")
print(f"Features: {len(feature_names)}")

Train: (5788, 40)  |  Test: (1447, 40)
Churn rate — train: 26.24%  |  test: 26.26%
Features: 40


## 🤖 Model Definitions

Define the candidate classifiers. `class_weight="balanced"` accounts for the ~26 % churn imbalance.

## 📊 Cross-Validation — Model Selection

In [ ]:
RANDOM_STATE = 42

imputer = SimpleImputer(strategy="median")

models = {
    "Logistic Regression (baseline)": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "clf",
                LogisticRegression(
                    max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
                ),
            ),
        ]
    ),
    "Random Forest": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=200, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
                ),
            ),
        ]
    ),
    "Gradient Boosting": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "clf",
                GradientBoostingClassifier(
                    n_estimators=200, learning_rate=0.1, max_depth=4, random_state=RANDOM_STATE
                ),
            ),
        ]
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["roc_auc", "f1", "precision", "recall"]

results = {}
for name, model in models.items():
    print(f"Training: {name} …", end="  ")
    cv_scores = cross_validate(model, x_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    results[name] = {
        "AUC-ROC": cv_scores["test_roc_auc"].mean(),
        "F1": cv_scores["test_f1"].mean(),
        "Precision": cv_scores["test_precision"].mean(),
        "Recall": cv_scores["test_recall"].mean(),
        "AUC_std": cv_scores["test_roc_auc"].std(),
    }
    print(f"AUC={results[name]['AUC-ROC']:.4f} ± {results[name]['AUC_std']:.4f}")

results_df = pd.DataFrame(results).T.sort_values("AUC-ROC", ascending=False)
print("\n--- 5-Fold CV Results ---")
print(results_df.to_string())

Training: Logistic Regression (baseline) …  

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/home/santi/Crum-CDPDN/.venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/santi/Crum-CDPDN/.venv/lib/python3.12/site-packages/sklearn/base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/santi/Crum-CDPDN/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py", line 1191, in fit
    X, y = validate_data(
           ^^^^^^^^^^^^^^
  File "/home/santi/Crum-CDPDN/.venv/lib/python3.12/site-packages/sklearn/utils/valida

In [ ]:
# Visual comparison
metrics = ["AUC-ROC", "F1", "Precision", "Recall"]
fig, axes = plt.subplots(1, len(metrics), figsize=(16, 4))

for ax, metric in zip(axes, metrics, strict=False):
    values = results_df[metric]
    bars = ax.barh(
        results_df.index,
        values,
        color=sns.color_palette("husl", len(results_df)),
        edgecolor="black",
        alpha=0.8,
    )
    ax.set_xlim(0, 1)
    ax.set_title(metric, fontsize=12, fontweight="bold")
    ax.set_xlabel("Score")
    ax.grid(axis="x", alpha=0.3)
    for bar, val in zip(bars, values, strict=False):
        ax.text(
            val + 0.01, bar.get_y() + bar.get_height() / 2, f"{val:.3f}", va="center", fontsize=9
        )

plt.suptitle("Model Comparison — 5-Fold Stratified CV", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 🏆 Train Best Model on Full Training Set

In [ ]:
best_model_name = results_df.index[0]
best_model = models[best_model_name]

print(f"Best model: {best_model_name}")
best_model.fit(x_train, y_train)

# Test-set evaluation
y_pred = best_model.predict(x_test)
y_proba = best_model.predict_proba(x_test)[:, 1]

print("\n--- Test Set Metrics ---")
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}")

### Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["No Churn", "Churn"], cmap="Blues", ax=axes[0]
)
axes[0].set_title(f"Confusion Matrix — {best_model_name}", fontweight="bold")

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1], name=best_model_name)
axes[1].plot([0, 1], [0, 1], "k--", label="Random")
axes[1].set_title("ROC Curve", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.show()

## 💾 Save Model & Predictions

In [ ]:
# Save the best model
model_path = MODELS_DIR / "best_model.pkl"
joblib.dump(best_model, model_path)

# Save metadata
metadata = {
    "model_name": best_model_name,
    "cv_results": results_df.to_dict(),
    "test_metrics": {
        "auc_roc": round(roc_auc_score(y_test, y_proba), 4),
        "f1": round(f1_score(y_test, y_pred), 4),
        "precision": round(precision_score(y_test, y_pred), 4),
        "recall": round(recall_score(y_test, y_pred), 4),
    },
}
with open(MODELS_DIR / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# Save predictions
pd.DataFrame({"y_true": y_test.values, "y_pred": y_pred, "y_proba": y_proba}).to_parquet(
    OUTPUT_DIR / "test_predictions.parquet", index=False
)

print("✅ Saved:")
print(f"   {model_path}")
print(f"   {MODELS_DIR / 'model_metadata.json'}")
print(f"   {OUTPUT_DIR / 'test_predictions.parquet'}")
print(f"\n📈 Final Test AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}")

## 📊 Analysis of Results and Conclusions

| Step | Output |
|---|---|
| 5-fold stratified CV | Compared Logistic Regression, Random Forest, Gradient Boosting |
| Best model selected | Highest AUC-ROC on validation folds |
| Test evaluation | Confusion matrix + ROC Curve |
| Artefacts saved | `best_model.pkl`, `model_metadata.json`, `test_predictions.parquet` |

**Key metric:** AUC-ROC is prioritised because the dataset is imbalanced (~26 % churn).  
`class_weight="balanced"` and stratified splits ensure fair evaluation.

## 💡 Proposals and Ideas

1. **Hyperparameter tuning**: use `RandomizedSearchCV` or `Optuna` on the winning model.  
2. **XGBoost / LightGBM**: add as candidates for potentially higher AUC-ROC.  
3. **Threshold optimisation**: adjust the decision threshold to maximise F1 or Recall for business needs.  
4. **Calibration**: apply `CalibratedClassifierCV` if calibrated probabilities are needed.

## 📖 References
- https://joserzapata.github.io/post/ciencia-datos-proyecto-python/5-models/
- Scikit-learn documentation: https://scikit-learn.org/stable/supervised_learning.html